# Resource Discovery and Informed Cluster Configuration

In this notebook, we will demonstrate how to use the CodeFlare SDK's enhanced resource discovery capabilities to:
 * Discover available resources in your workspace
 * Understand resource quotas and flavors
 * Make informed decisions about cluster configuration
 * Choose appropriate resources for RayJobs
 * Monitor queue utilization


## Import Required Packages

First, let's import the necessary CodeFlare SDK packages including the new resource discovery functions.


In [ ]:
from codeflare_sdk import (
    Cluster, 
    ClusterConfiguration, 
    RayJob, 
    TokenAuthentication,
    get_available_resources_summary,
    get_queue_resource_info,
    get_resource_flavor_details,
    get_cluster_queue_details,
    list_local_queues,
    analyze_queue_utilization,
    find_best_queue_for_workload
)


## Authentication

Set up authentication for accessing the cluster resources.


In [ ]:
# Create authentication object
auth = TokenAuthentication(
    token="XXXXX",  # Replace with your actual token
    server="XXXXX",  # Replace with your actual server URL
    skip_tls=False
)
auth.login()


## Basic Resource Discovery

Let's start by discovering what resources are available in your workspace.


In [ ]:
# List all available local queues
print("=== Available Local Queues ===")
queues = list_local_queues()
print(f"Found {len(queues)} local queues:")
for queue in queues:
    print(f"  - {queue['name']}")
    if 'flavors' in queue:
        print(f"    Flavors: {queue['flavors']}")


In [ ]:
# Get a high-level summary of available resources
print("\n=== Resource Summary ===")
summary = get_available_resources_summary()
print(f"Namespace: {summary['namespace']}")
print(f"Total queues: {summary['total_queues']}")
print(f"Default queue: {summary['default_queue']}")
print(f"\nAvailable resources:")
print(f"  CPU: {summary['available_resources']['cpu']}")
print(f"  Memory: {summary['available_resources']['memory']}")
print(f"  GPU: {summary['available_resources']['gpu']}")
print(f"\nResource types: {summary['resource_types']}")
print(f"Flavors: {summary['flavors']}")


## Intelligent Resource Selection

Let's create a function to help choose the best resources for different types of workloads.


In [ ]:
# Test the native SDK function with ClusterConfiguration support
print("=== Finding Best Resources with ClusterConfiguration Support ===")

# Example 1: Using ClusterConfiguration object
print("\n1. Using ClusterConfiguration object:")
cluster_config = ClusterConfiguration(
    name='test-cluster',
    num_workers=3,
    worker_cpu_requests='1',
    worker_cpu_limits='1',
    worker_memory_requests=8,
    worker_memory_limits=8,
    worker_extended_resource_requests={'nvidia.com/gpu': 1},
    worker_extended_resource_limits={'nvidia.com/gpu': 1}
)

# The function will extract resource requirements from the ClusterConfiguration
result = find_best_queue_for_workload(cluster_config=cluster_config)

# Example 2: Traditional individual parameters (still supported)
print("\n2. Traditional individual parameters:")
result = find_best_queue_for_workload(cpu_required=2, memory_required=8, gpu_required=0)

# Example 3: Mixed usage (ClusterConfiguration takes precedence)
print("\n3. Mixed usage (ClusterConfiguration takes precedence):")
light_config = ClusterConfiguration(
    name='light-cluster',
    num_workers=2,
    worker_cpu_requests='500m',
    worker_cpu_limits='500m',
    worker_memory_requests=2,
    worker_memory_limits=2
)

# Individual parameters will be ignored in favor of cluster_config
result = find_best_queue_for_workload(
    cluster_config=light_config,
    cpu_required=10,  # This will be ignored
    memory_required=100,  # This will be ignored
    verbose=False
)

if result['success']:
    print(f"   ✅ Used cluster_config values: {result['queue_name']}")
    print(f"   📊 Total resources analyzed: {result['available_resources']['cpu']} CPU, {result['available_resources']['memory']}GB RAM")


In [ ]:
# Test the function with different workload requirements
print("=== Finding Best Resources for Different Workloads ===")

# CPU-intensive workload
queue, flavor, resources = find_best_queue_for_workload(cpu_required=2, memory_required=8, gpu_required=0)
if queue:
    print(f"\nCPU-intensive workload (2 CPU, 8GB RAM):")
    print(f"  Recommended queue: {queue}")
    print(f"  Recommended flavor: {flavor}")
    print(f"  Available: {resources['cpu_available']} CPU, {resources['memory_available']}GB RAM")
    print(f"  Queue utilization: {resources['utilization_score']} active workloads")
else:
    print("\nNo suitable resources found for CPU-intensive workload")

# GPU workload
queue, flavor, resources = find_best_queue_for_workload(cpu_required=4, memory_required=16, gpu_required=1)
if queue:
    print(f"\nGPU workload (4 CPU, 16GB RAM, 1 GPU):")
    print(f"  Recommended queue: {queue}")
    print(f"  Recommended flavor: {flavor}")
    print(f"  Available: {resources['cpu_available']} CPU, {resources['memory_available']}GB RAM, {resources['gpu_available']} GPU")
    print(f"  Queue utilization: {resources['utilization_score']} active workloads")
else:
    print("\nNo suitable resources found for GPU workload")


## Informed Cluster Configuration

Now let's create clusters using the resource discovery information to make informed decisions.


In [ ]:
# Create a cluster using ClusterConfiguration with intelligent queue selection
print("=== Creating Cluster with ClusterConfiguration Analysis ===")

# Define your cluster configuration
cluster_config = ClusterConfiguration(
    name='informed-cluster',
    num_workers=2,
    head_cpu_requests='500m',
    head_cpu_limits='500m',
    head_memory_requests=2,
    head_memory_limits=2,
    worker_cpu_requests='1',
    worker_cpu_limits='1',
    worker_memory_requests=4,
    worker_memory_limits=4,
    write_to_file=False
)

# Find the best queue for this configuration
result = find_best_queue_for_workload(cluster_config=cluster_config)

if result['success']:
    # Update the cluster configuration with the recommended queue
    if result['cluster_config_suggestions']['local_queue']:
        cluster_config.local_queue = result['cluster_config_suggestions']['local_queue']
        print(f"\n📊 Updated cluster configuration:")
        print(f"  Name: {cluster_config.name}")
        print(f"  Queue: {cluster_config.local_queue}")
        print(f"  Workers: {cluster_config.num_workers}")
        print(f"  Worker CPU: {cluster_config.worker_cpu_requests}")
        print(f"  Worker Memory: {cluster_config.worker_memory_requests}GB")
    
    # Create and apply the cluster
    cluster = Cluster(cluster_config)
    cluster.apply()
    
    print("\n🚀 Cluster submitted! Check status below:")
    cluster.status()
    
else:
    print("\n❌ Cannot create cluster - no suitable resources found")
    print(f"Reason: {result['reason']}")
    if result['recommendations']:
        print(f"Suggestions: {', '.join(result['recommendations'][:2])}")


## Monitor Cluster Status

Let's monitor the cluster and wait for it to be ready.


In [ ]:
# Monitor cluster status
print("=== Monitoring Cluster Status ===")
cluster.status()

print("\nWaiting for cluster to be ready...")
cluster.wait_ready()

print("\nCluster is ready! Final status:")
cluster.status()


## Informed RayJob Configuration

Now let's create a RayJob using the same resource discovery principles.


In [ ]:
# Create a RayJob with informed resource selection
print("=== Creating RayJob with Informed Resource Selection ===")

# Check current queue utilization before submitting job
queue_info = get_queue_resource_info()
current_queue = None

for queue_name, queue_data in queue_info["queues"].items():
    if queue_data["is_default"]:
        current_queue = queue_name
        print(f"Using default queue: {queue_name}")
        print(f"  Pending workloads: {queue_data['status']['pending']}")
        print(f"  Running workloads: {queue_data['status']['running']}")
        break

if not current_queue:
    # Use the first available queue
    current_queue = list(queue_info["queues"].keys())[0]
    print(f"Using first available queue: {current_queue}")

# Create RayJob with appropriate runtime environment based on available resources
rayjob = RayJob(
    job_name="informed-rayjob",
    cluster_name="informed-cluster",  # Use our existing cluster
    namespace="default",  # Adjust as needed
    entrypoint="python -c \"import ray; ray.init(); print('Hello from informed RayJob!'); import time; time.sleep(30); print('RayJob completed successfully!')\"",
    runtime_env={
        "pip": ["torch", "numpy", "pandas"],  # Common ML libraries
        "env_vars": {
            "WORKLOAD_TYPE": "cpu_intensive",
            "RESOURCE_DISCOVERY": "enabled"
        }
    },
    shutdown_after_job_finishes=False,  # Keep cluster running
    ttl_seconds_after_finished=300,  # Clean up job after 5 minutes
)

print(f"\nRayJob configuration:")
print(f"  Job name: informed-rayjob")
print(f"  Cluster: informed-cluster")
print(f"  Runtime environment: torch, numpy, pandas")
print(f"  Shutdown after job: False")

# Submit the job
print("\nSubmitting RayJob...")
submission_result = rayjob.submit()
print(f"RayJob submitted successfully: {submission_result}")


## Monitor RayJob Status

Let's monitor the RayJob status and see how it progresses.


## Advanced Queue Analysis and Programmatic Usage

Let's explore more advanced usage patterns and programmatic integration scenarios.


In [ ]:
# Advanced Queue Analysis with Programmatic Usage
print("=== Advanced Queue Analysis and Programmatic Usage ===\n")

# Example 1: Programmatic queue selection for different workload types
print("1. Comparing multiple workload types:")
workloads = [
    {"name": "Light", "cpu": 1, "memory": 2, "gpu": 0},
    {"name": "Medium", "cpu": 2, "memory": 8, "gpu": 0},
    {"name": "Heavy", "cpu": 4, "memory": 16, "gpu": 0},
    {"name": "GPU", "cpu": 2, "memory": 8, "gpu": 1},
]

print("Workload Analysis (Silent Mode):")
for workload in workloads:
    result = find_best_queue_for_workload(
        cpu_required=workload["cpu"],
        memory_required=workload["memory"],
        gpu_required=workload["gpu"],
        verbose=False  # Silent mode for programmatic use
    )
    
    if result['success']:
        print(f"  {workload['name']:6}: {result['queue_name']} (util: {result['utilization_score']}, default: {result['is_default']})")
    else:
        print(f"  {workload['name']:6}: No suitable queue")

print("\n" + "="*60 + "\n")

# Example 2: Programmatic safety checks
print("2. Programmatic safety checks:")

def is_safe_to_submit_workload():
    """Check if it's safe to submit a new workload"""
    analysis = analyze_queue_utilization()
    
    if analysis['analysis']['total_queues'] == 0:
        return False, "No queues available"
    
    # Check for critical queues
    critical_count = len([q for q in analysis['queues'] if q['utilization_level'] == 'critical'])
    if critical_count == analysis['analysis']['total_queues']:
        return False, "All queues are critically utilized"
    
    # Check for high utilization
    high_count = len([q for q in analysis['queues'] if q['utilization_level'] == 'high'])
    if high_count > analysis['analysis']['total_queues'] // 2:
        return False, "Most queues have high utilization"
    
    return True, "Safe to submit new workloads"

def find_best_queue_for_new_workload():
    """Find the best queue for a new workload based on analysis"""
    analysis = analyze_queue_utilization()
    
    if not analysis['queues']:
        return None, "No queues available"
    
    # Filter out critical queues
    available_queues = [q for q in analysis['queues'] if q['utilization_level'] != 'critical']
    
    if not available_queues:
        return None, "All queues are critically utilized"
    
    # Find the least utilized available queue
    best_queue = min(available_queues, key=lambda x: x['total'])
    
    return best_queue['name'], f"Recommended: {best_queue['name']} ({best_queue['utilization_level']} utilization)"

# Demonstrate the functions
best_queue, recommendation = find_best_queue_for_new_workload()
print(f"Best queue for new workload: {best_queue}")
print(f"Recommendation: {recommendation}")

is_safe, safety_message = is_safe_to_submit_workload()
print(f"\nSafe to submit workload: {is_safe}")
print(f"Safety assessment: {safety_message}")


## Comprehensive Resource Discovery Testing

Let's test all the resource discovery functions to ensure they work correctly and demonstrate their capabilities.


In [ ]:
# Comprehensive Resource Discovery Testing
print("=== CodeFlare SDK Enhanced Resource Discovery Test ===\n")

# Test 1: Basic queue listing (existing functionality)
print("1. Testing basic queue listing...")
try:
    queues = list_local_queues()
    print(f"   Found {len(queues)} local queues:")
    for queue in queues:
        print(f"     - {queue['name']}")
        if 'flavors' in queue:
            print(f"       Flavors: {queue['flavors']}")
except Exception as e:
    print(f"   Error: {e}")

print()

# Test 2: Resource summary
print("2. Testing resource summary...")
try:
    summary = get_available_resources_summary()
    print(f"   Namespace: {summary['namespace']}")
    print(f"   Total queues: {summary['total_queues']}")
    print(f"   Default queue: {summary['default_queue']}")
    print(f"   Available resources: {summary['available_resources']}")
    print(f"   Resource types: {summary['resource_types']}")
    print(f"   Flavors: {summary['flavors']}")
except Exception as e:
    print(f"   Error: {e}")

print()

# Test 3: Detailed queue information
print("3. Testing detailed queue information...")
try:
    queue_info = get_queue_resource_info()
    
    if queue_info["queues"]:
        print(f"   Found {len(queue_info['queues'])} queues with detailed info:")
        
        for queue_name, queue_data in queue_info["queues"].items():
            print(f"     Queue: {queue_name}")
            print(f"       Cluster Queue: {queue_data['cluster_queue']}")
            print(f"       Is Default: {queue_data['is_default']}")
            print(f"       Status: {queue_data['status']}")
            
            for flavor in queue_data["flavors"]:
                print(f"       Flavor: {flavor['name']}")
                for resource_type, resource_info in flavor["resources"].items():
                    print(f"         {resource_type}: {resource_info['nominalQuota']}")
    else:
        print("   No queues found")
        
except Exception as e:
    print(f"   Error: {e}")

print()

# Test 4: Resource flavor details (if any flavors exist)
print("4. Testing resource flavor details...")
try:
    summary = get_available_resources_summary()
    if summary['flavors']:
        flavor_name = summary['flavors'][0]  # Test first flavor
        flavor_info = get_resource_flavor_details(flavor_name)
        print(f"   Flavor: {flavor_info['name']}")
        print(f"   Node Labels: {flavor_info['node_labels']}")
        print(f"   Tolerations: {len(flavor_info['tolerations'])} tolerations")
    else:
        print("   No flavors found to test")
except Exception as e:
    print(f"   Error: {e}")

print()

# Test 5: Cluster queue details (if any cluster queues exist)
print("5. Testing cluster queue details...")
try:
    queue_info = get_queue_resource_info()
    if queue_info["queues"]:
        # Get first queue's cluster queue
        first_queue = list(queue_info["queues"].values())[0]
        cluster_queue_name = first_queue["cluster_queue"]
        
        if cluster_queue_name:
            cluster_queue_info = get_cluster_queue_details(cluster_queue_name)
            print(f"   Cluster Queue: {cluster_queue_info['name']}")
            print(f"   Has Spec: {'spec' in cluster_queue_info}")
            print(f"   Has Status: {'status' in cluster_queue_info}")
        else:
            print("   No cluster queue found")
    else:
        print("   No queues found to test cluster queue details")
except Exception as e:
    print(f"   Error: {e}")

print("\n=== Test Complete ===")


## Practical Usage Scenarios

Let's demonstrate practical scenarios for finding suitable resources and checking queue utilization.


In [ ]:
# Practical Usage Scenarios
print("=== Practical Usage Demonstration ===\n")

# Scenario 1: Find suitable resources for a workload
print("Scenario 1: Finding suitable resources for a CPU-intensive workload...")
try:
    queue_info = get_queue_resource_info()
    
    cpu_required = 2
    memory_required = 8
    
    suitable_queues = []
    
    for queue_name, queue_data in queue_info["queues"].items():
        for flavor in queue_data["flavors"]:
            resources = flavor["resources"]
            
            cpu_available = float(resources.get("cpu", {}).get("nominalQuota", "0"))
            memory_available = float(resources.get("memory", {}).get("nominalQuota", "0").replace("Gi", ""))
            
            if cpu_available >= cpu_required and memory_available >= memory_required:
                suitable_queues.append({
                    "queue": queue_name,
                    "flavor": flavor["name"],
                    "cpu": cpu_available,
                    "memory": memory_available
                })
    
    if suitable_queues:
        print(f"   Found {len(suitable_queues)} suitable queues:")
        for sq in suitable_queues:
            print(f"     - Queue: {sq['queue']}, Flavor: {sq['flavor']}")
            print(f"       CPU: {sq['cpu']}, Memory: {sq['memory']}Gi")
    else:
        print("   No suitable queues found for the requirements")
        
except Exception as e:
    print(f"   Error: {e}")

print()

# Scenario 2: Check queue utilization
print("Scenario 2: Checking queue utilization...")
try:
    queue_info = get_queue_resource_info()
    
    for queue_name, queue_data in queue_info["queues"].items():
        status = queue_data["status"]
        print(f"   Queue: {queue_name}")
        print(f"     Pending: {status['pending']}")
        print(f"     Admitted: {status['admitted']}")
        print(f"     Running: {status['running']}")
        
        if status['pending'] > 0:
            print(f"     ⚠️  Queue has {status['pending']} pending workloads")
        else:
            print(f"     ✅ Queue is available")
            
except Exception as e:
    print(f"   Error: {e}")

print()

# Scenario 3: Advanced queue selection with constraints
print("Scenario 3: Advanced queue selection with constraints...")

# Example: Find queue avoiding critical utilization
print("Finding queue avoiding critical utilization:")
result = find_best_queue_for_workload(
    cpu_required=1, 
    memory_required=4, 
    gpu_required=0,
    avoid_critical=True
)

if result['success']:
    print(f"   ✅ Selected queue: {result['queue_name']}")
    print(f"   ✅ Utilization: {result['utilization_score']} workloads")
    print(f"   ✅ Avoided critical queues: {len(result['warnings'])} warnings")
else:
    print(f"   ❌ No non-critical queues available")

print()

# Example: Prefer default queue
print("Finding queue preferring default queue:")
result = find_best_queue_for_workload(
    cpu_required=1, 
    memory_required=2, 
    gpu_required=0,
    prefer_default=True
)

if result['success']:
    print(f"   ✅ Selected queue: {result['queue_name']}")
    print(f"   ✅ Is default queue: {result['is_default']}")
    print(f"   ✅ Utilization: {result['utilization_score']} workloads")
else:
    print(f"   ❌ No suitable queue found")


## Standalone Script Examples

Here are examples of how to use these functions in standalone Python scripts outside of Jupyter notebooks.


In [ ]:
# Standalone Script Examples
print("=== Standalone Script Examples ===\n")

print("Example 1: Basic Queue Analysis Script")
print("```python")
print("#!/usr/bin/env python3")
print('"""')
print("Basic queue analysis script")
print('"""')
print("")
print("from codeflare_sdk import analyze_queue_utilization")
print("")
print("def main():")
print("    analysis = analyze_queue_utilization()")
print("    ")
print("    print(f\"Total queues: {analysis['analysis']['total_queues']}\")")
print("    print(f\"Total workloads: {analysis['analysis']['total_workloads']}\")")
print("    print(f\"Primary recommendation: {analysis['recommendations']['primary_suggestion']}\")")
print("    ")
print("    for queue in analysis['queues']:")
print("        print(f\"{queue['name']}: {queue['total']} workloads ({queue['utilization_level']})\")")
print("")
print("if __name__ == '__main__':")
print("    main()")
print("```")

print("\n" + "="*60 + "\n")

print("Example 2: Intelligent Cluster Creation Script")
print("```python")
print("#!/usr/bin/env python3")
print('"""')
print("Intelligent cluster creation script")
print('"""')
print("")
print("from codeflare_sdk import Cluster, ClusterConfiguration, find_best_queue_for_workload")
print("")
print("def create_intelligent_cluster(name, workers, cpu_per_worker, memory_per_worker):")
print("    # Find best queue for the workload")
print("    result = find_best_queue_for_workload(")
print("        cpu_required=cpu_per_worker * workers,")
print("        memory_required=memory_per_worker * workers,")
print("        gpu_required=0,")
print("        verbose=False")
print("    )")
print("    ")
print("    if not result['success']:")
print("        print(f\"Cannot create cluster: {result['reason']}\")")
print("        return None")
print("    ")
print("    # Create cluster configuration")
print("    cluster_config = ClusterConfiguration(")
print("        name=name,")
print("        local_queue=result['queue_name'],")
print("        num_workers=workers,")
print("        worker_cpu_requests=str(cpu_per_worker),")
print("        worker_cpu_limits=str(cpu_per_worker),")
print("        worker_memory_requests=memory_per_worker,")
print("        worker_memory_limits=memory_per_worker,")
print("        write_to_file=False")
print("    )")
print("    ")
print("    # Create and apply cluster")
print("    cluster = Cluster(cluster_config)")
print("    cluster.apply()")
print("    ")
print("    print(f\"Cluster '{name}' created using queue '{result['queue_name']}'\"")
print("    return cluster")
print("")
print("if __name__ == '__main__':")
print("    cluster = create_intelligent_cluster('my-cluster', 2, 1, 4)")
print("```")

print("\n" + "="*60 + "\n")

print("Example 3: Queue Monitoring Script")
print("```python")
print("#!/usr/bin/env python3")
print('"""')
print("Queue monitoring script")
print('"""')
print("")
print("import time")
print("from codeflare_sdk import analyze_queue_utilization")
print("")
print("def monitor_queues(interval=60):")
print("    \"\"\"Monitor queue utilization every interval seconds\"\"\"")
print("    while True:")
print("        analysis = analyze_queue_utilization()")
print("        ")
print("        print(f\"\\n[{time.strftime('%H:%M:%S')}] Queue Status:\")")
print("        for queue in analysis['queues']:")
print("            status = f\"{queue['pending']}P/{queue['running']}R\"")
print("            level = queue['utilization_level']")
print("            print(f\"  {queue['name']}: {status} ({level})\")")
print("        ")
print("        # Check for critical queues")
print("        critical = [q for q in analysis['queues'] if q['utilization_level'] == 'critical']")
print("        if critical:")
print("            print(f\"  ⚠️ Critical queues: {[q['name'] for q in critical]}\")")
print("        ")
print("        time.sleep(interval)")
print("")
print("if __name__ == '__main__':")
print("    monitor_queues()")
print("```")

print("\n" + "="*60 + "\n")

print("Example 4: Workload Safety Check Script")
print("```python")
print("#!/usr/bin/env python3")
print('"""')
print("Workload safety check script")
print('"""')
print("")
print("from codeflare_sdk import analyze_queue_utilization")
print("")
print("def check_workload_safety():")
print("    \"\"\"Check if it's safe to submit new workloads\"\"\"")
print("    analysis = analyze_queue_utilization()")
print("    ")
print("    if analysis['analysis']['total_queues'] == 0:")
print("        return False, \"No queues available\"")
print("    ")
print("    # Check for critical queues")
print("    critical_count = len([q for q in analysis['queues'] if q['utilization_level'] == 'critical'])")
print("    if critical_count == analysis['analysis']['total_queues']:")
print("        return False, \"All queues are critically utilized\"")
print("    ")
print("    # Check for high utilization")
print("    high_count = len([q for q in analysis['queues'] if q['utilization_level'] == 'high'])")
print("    if high_count > analysis['analysis']['total_queues'] // 2:")
print("        return False, \"Most queues have high utilization\"")
print("    ")
print("    return True, \"Safe to submit new workloads\"")
print("")
print("if __name__ == '__main__':")
print("    is_safe, message = check_workload_safety()")
print("    print(f\"Safe to submit: {is_safe}\")")
print("    print(f\"Message: {message}\")")
print("```")


In [ ]:
# Monitor RayJob status
print("=== Monitoring RayJob Status ===")

import time

max_attempts = 10
attempt = 0

while attempt < max_attempts:
    attempt += 1
    print(f"\nAttempt {attempt}/{max_attempts}:")
    
    try:
        status = rayjob.status()
        print(f"RayJob status: {status}")
        
        # Check if job is completed
        if "COMPLETED" in str(status) or "FAILED" in str(status):
            print(f"\nRayJob finished with status: {status}")
            break
            
    except Exception as e:
        print(f"Error checking status: {e}")
    
    if attempt < max_attempts:
        print("Waiting 30 seconds before next check...")
        time.sleep(30)

if attempt >= max_attempts:
    print("\nMax monitoring attempts reached. Check job status manually.")


## Resource Monitoring and Optimization

Let's demonstrate how to monitor resource utilization and optimize future workloads.


In [ ]:
# Monitor resource utilization across queues
print("=== Resource Utilization Monitoring ===")

# Use the native SDK function for queue utilization analysis
utilization_analysis = analyze_queue_utilization()

print(f"\n📊 Analysis Summary:")
print(f"  Total queues: {utilization_analysis['analysis']['total_queues']}")
print(f"  Total workloads: {utilization_analysis['analysis']['total_workloads']}")
print(f"  Average utilization: {utilization_analysis['analysis']['average_utilization']}")
print(f"  Most utilized: {utilization_analysis['analysis']['most_utilized_queue']}")
print(f"  Least utilized: {utilization_analysis['analysis']['least_utilized_queue']}")

print(f"\n📋 Queue Details:")
for queue in utilization_analysis['queues']:
    print(f"  {queue['name']}:")
    print(f"    Pending: {queue['pending']}, Admitted: {queue['admitted']}, Running: {queue['running']}")
    print(f"    Total: {queue['total']}, Level: {queue['utilization_level']}")
    print(f"    {queue['recommendation']}")
    if queue['is_default']:
        print(f"    🎯 Default queue")

print(f"\n🎯 Primary Recommendation:")
print(f"  {utilization_analysis['recommendations']['primary_suggestion']}")

if utilization_analysis['recommendations']['warnings']:
    print(f"\n⚠️ Warnings:")
    for warning in utilization_analysis['recommendations']['warnings']:
        print(f"  {warning}")

if utilization_analysis['recommendations']['suggestions']:
    print(f"\n💡 Suggestions:")
    for suggestion in utilization_analysis['recommendations']['suggestions']:
        print(f"  {suggestion}")


## Cleanup

Finally, let's clean up our resources.


In [ ]:
# Clean up resources
print("=== Cleaning Up Resources ===")

# Take down the cluster
print("Taking down cluster...")
cluster.down()

print("\nFinal resource utilization check:")
final_summary = get_available_resources_summary()
print(f"Available resources after cleanup:")
print(f"  CPU: {final_summary['available_resources']['cpu']}")
print(f"  Memory: {final_summary['available_resources']['memory']}")
print(f"  GPU: {final_summary['available_resources']['gpu']}")

print("\n✅ Cleanup completed!")


## Conclusion

This notebook demonstrated the comprehensive capabilities of the CodeFlare SDK's enhanced resource discovery system:

### **Core Functionality Covered:**

1. **Resource Discovery**: 
   - `get_available_resources_summary()` - High-level resource overview
   - `get_queue_resource_info()` - Detailed queue and flavor information
   - `get_resource_flavor_details()` - Specific flavor characteristics
   - `get_cluster_queue_details()` - Cluster queue specifications

2. **Intelligent Analysis**:
   - `analyze_queue_utilization()` - Comprehensive queue utilization analysis with recommendations
   - `find_best_queue_for_workload()` - Smart queue selection based on resource requirements

3. **Advanced Features**:
   - **ClusterConfiguration Integration**: Automatic resource extraction from cluster configs
   - **Verbose Output Control**: Built-in formatted output with silent mode option
   - **Programmatic Usage**: Functions designed for both interactive and script usage
   - **Safety Checks**: Built-in validation and warning systems

### **Usage Patterns Demonstrated:**

1. **Interactive Notebook Usage**: Step-by-step exploration and analysis
2. **Programmatic Integration**: Silent mode for automated workflows
3. **Standalone Scripts**: Complete examples for production use
4. **Advanced Scenarios**: Constraint-based selection and safety checks
5. **Comprehensive Testing**: Full API validation and error handling

### **Key Benefits:**

- **Self-Service Resource Discovery**: No administrator intervention needed
- **Intelligent Resource Selection**: AI-powered queue and flavor recommendations
- **Error Prevention**: Validation before workload submission
- **Queue Optimization**: Avoid over-subscribed queues
- **Informed Configuration**: Data-driven cluster and job setup
- **Production Ready**: Comprehensive error handling and edge cases

### **Integration Points:**

- **Cluster Creation**: Seamless integration with `ClusterConfiguration`
- **RayJob Submission**: Informed resource selection for jobs
- **Monitoring**: Real-time queue utilization tracking
- **Automation**: Programmatic resource management

### **Next Steps:**

- Use these functions in your production workflows
- Implement automated resource selection based on workload requirements
- Monitor resource utilization regularly for optimization opportunities
- Explore detailed flavor information for node characteristics and constraints
- Consider implementing custom resource selection logic based on your specific needs

### **Standalone Usage:**

All functionality is available as standalone Python scripts, making it easy to integrate into:
- CI/CD pipelines
- Automated cluster management
- Resource monitoring systems
- Custom workload schedulers
